# ML-08 — Capstone Modeling Lane

This notebook compares a readable model with the Week 4 rule on the same held-out clients.


## 1. Method choice

The target is a yes-or-no June impression decline, and the practical output is a ranked list for review. Logistic regression is a readable first model for five numeric May features. Its probability is used only as a ranking score. The model does not receive client IDs, content IDs, June outcomes, query metrics, or product flags.

## Load the May and June data

I keep the large daily records in DuckDB and bring only page-level results into pandas. The first run saves the results locally. The daily batches keep memory use down.


In [1]:
import getpass
import os
import tempfile
from datetime import date, timedelta
from pathlib import Path
import duckdb
import numpy as np
import pandas as pd
from sklearn.model_selection import GroupShuffleSplit

FEATURE_COLUMNS = (
    'log_prior30_impressions', 'log_prior30_clicks', 'prior30_ctr',
    'prior30_avg_position_filled', 'position_missing',
)

LABEL_END = date(2026, 6, 30)
HISTORICAL_CUTOFF = LABEL_END - timedelta(days=30)
FEATURE_START = HISTORICAL_CUTOFF - timedelta(days=29)
CURRENT_START = HISTORICAL_CUTOFF + timedelta(days=1)

def repo_root() -> Path:
    for path in (Path.cwd(), *Path.cwd().parents):
        if (path / 'work' / 'notebooks').is_dir():
            return path
    raise FileNotFoundError('Open the repository or a folder inside it before running the notebook')

def _quote_path(path: Path) -> str:
    return str(path).replace('\\', '/').replace("'", "''")

def _feature_select(table: str, labeled: bool) -> str:
    extra = (
        ", future30_impressions, CASE WHEN future30_impressions < 0.8 * prior30_impressions "
        "THEN 1 ELSE 0 END AS future_decline_label"
    ) if labeled else ''
    return f"""
        SELECT client_hash_id, content_hash_id, prior30_impressions, prior30_clicks,
               prior30_avg_position,
               LN(1 + prior30_impressions) AS log_prior30_impressions,
               LN(1 + prior30_clicks) AS log_prior30_clicks,
               CAST(prior30_clicks AS DOUBLE) / prior30_impressions AS prior30_ctr,
               COALESCE(prior30_avg_position, 0.0) AS prior30_avg_position_filled,
               CASE WHEN prior30_avg_position IS NULL THEN 1 ELSE 0 END AS position_missing
               {extra}
        FROM {table}
    """

repo = repo_root()


In [2]:
def _summarize_month(con: duckdb.DuckDBPyConnection, source: str,
                     start: date, end: date, folder: Path) -> Path:
    """Stream a month to disk, then deduplicate one day at a time."""
    folder.mkdir()
    stage = folder / 'daily_source.parquet'
    days = folder / 'days'
    days.mkdir()
    summary = folder / 'summary.parquet'
    con.execute(f"""
        COPY (SELECT report_date, client_hash_id, content_hash_id,
                     gsc_impressions, gsc_clicks, gsc_avg_position
              FROM {source}
              WHERE report_date BETWEEN DATE '{start}' AND DATE '{end}')
        TO '{_quote_path(stage)}' (FORMAT PARQUET)
    """)
    day = start
    while day <= end:
        day_path = days / f'{day}.parquet'
        con.execute(f"""
            COPY (
                SELECT client_hash_id, content_hash_id,
                       MIN(gsc_impressions) AS impressions,
                       MIN(gsc_clicks) AS clicks,
                       CASE WHEN MIN(gsc_avg_position) > 0 AND MIN(gsc_impressions) > 0
                            THEN MIN(gsc_avg_position) * MIN(gsc_impressions)
                            ELSE 0 END AS position_numerator,
                       CASE WHEN MIN(gsc_avg_position) > 0 AND MIN(gsc_impressions) > 0
                            THEN MIN(gsc_impressions) ELSE 0 END AS position_denominator,
                       COUNT(*) - 1 AS repeated_rows,
                       CASE WHEN MIN(gsc_impressions) IS DISTINCT FROM MAX(gsc_impressions)
                                      OR MIN(gsc_clicks) IS DISTINCT FROM MAX(gsc_clicks)
                                      OR MIN(gsc_avg_position) IS DISTINCT FROM MAX(gsc_avg_position)
                                      OR (COUNT(gsc_impressions) > 0 AND COUNT(gsc_impressions) < COUNT(*))
                                      OR (COUNT(gsc_clicks) > 0 AND COUNT(gsc_clicks) < COUNT(*))
                                      OR (COUNT(gsc_avg_position) > 0 AND COUNT(gsc_avg_position) < COUNT(*))
                            THEN 1 ELSE 0 END AS conflicting_keys
                FROM read_parquet('{_quote_path(stage)}')
                WHERE report_date = DATE '{day}'
                GROUP BY client_hash_id, content_hash_id
            ) TO '{_quote_path(day_path)}' (FORMAT PARQUET)
        """)
        day += timedelta(days=1)
    stage.unlink()
    con.execute(f"""
        COPY (
            SELECT client_hash_id, content_hash_id, COUNT(*) AS observed_days,
                   COUNT(*) FILTER (WHERE impressions IS NULL OR clicks IS NULL) AS missing_days,
                   SUM(impressions) AS impressions, SUM(clicks) AS clicks,
                   SUM(position_numerator) / NULLIF(SUM(position_denominator), 0) AS avg_position,
                   SUM(repeated_rows) AS repeated_rows,
                   SUM(conflicting_keys) AS conflicting_keys
            FROM read_parquet('{_quote_path(days / '*.parquet')}')
            GROUP BY client_hash_id, content_hash_id
        ) TO '{_quote_path(summary)}' (FORMAT PARQUET)
    """)
    for day_path in days.glob('*.parquet'):
        day_path.unlink()
    days.rmdir()
    return summary


In [3]:
def build_frames(source_override: str | None = None,
                 connection: duckdb.DuckDBPyConnection | None = None
                 ) -> tuple[pd.DataFrame, pd.DataFrame]:
    output = repo_root() / 'work' / 'outputs'
    output.mkdir(parents=True, exist_ok=True)
    con = connection or duckdb.connect()
    with tempfile.TemporaryDirectory(prefix='lane2_build_', dir=output) as scratch:
        temp = Path(scratch)
        spill = output / 'lane2_duckdb_spill'
        spill.mkdir(exist_ok=True)
        con.execute(f"SET temp_directory = '{_quote_path(spill)}'")
        con.execute("SET memory_limit = '1GB'")
        con.execute('SET threads = 2')
        if source_override is None:
            token = os.environ.get('HF_TOKEN') or getpass.getpass('Hugging Face READ token: ')
            con.execute("CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '"
                        + token.replace("'", "''") + "')")
            del token
            root = 'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance'
            may_source = f"read_parquet('{root}/month=2026-05/*.parquet')"
            june_source = f"read_parquet('{root}/month=2026-06/*.parquet')"
        else:
            may_source = june_source = source_override

        may = _summarize_month(con, may_source, FEATURE_START, HISTORICAL_CUTOFF,
                               temp / 'may')
        june = _summarize_month(con, june_source, CURRENT_START, LABEL_END,
                                temp / 'june')
        may_rel = f"read_parquet('{_quote_path(may)}')"
        june_rel = f"read_parquet('{_quote_path(june)}')"
        counts = con.sql(f"""
            SELECT SUM(repeated_rows), SUM(conflicting_keys),
                   COUNT(*) FILTER (WHERE client_hash_id IS NULL OR content_hash_id IS NULL)
            FROM (SELECT * FROM {may_rel} UNION ALL SELECT * FROM {june_rel})
        """).fetchone()
        repeated, conflicting, missing_keys = (int(value or 0) for value in counts)
        assert missing_keys == 0, f'{missing_keys:,} rows have a missing daily key'
        assert conflicting == 0, f'{conflicting:,} daily keys have different GSC values'

        con.execute(f"""
            CREATE OR REPLACE TEMP TABLE historical_metrics AS
            SELECT m.client_hash_id, m.content_hash_id,
                   m.impressions AS prior30_impressions,
                   m.clicks AS prior30_clicks,
                   m.avg_position AS prior30_avg_position,
                   j.impressions AS future30_impressions
            FROM {may_rel} m
            JOIN {june_rel} j USING (client_hash_id, content_hash_id)
            WHERE m.observed_days = 30 AND m.missing_days = 0
              AND m.impressions >= 100
              AND j.observed_days = 30 AND j.missing_days = 0
        """)
        con.execute(f"""
            CREATE OR REPLACE TEMP TABLE current_metrics AS
            SELECT client_hash_id, content_hash_id,
                   impressions AS prior30_impressions,
                   clicks AS prior30_clicks,
                   avg_position AS prior30_avg_position
            FROM {june_rel}
            WHERE observed_days = 30 AND missing_days = 0 AND impressions >= 100
        """)
        historical = con.sql(_feature_select('historical_metrics', labeled=True)).df()
        current = con.sql(_feature_select('current_metrics', labeled=False)).df()

    assert len(historical) > 0 and len(current) > 0, 'No pages met the coverage and impression rules'
    assert historical[list(FEATURE_COLUMNS)].notna().all().all()
    assert current[list(FEATURE_COLUMNS)].notna().all().all()
    print(f'May-June repeated GSC rows removed: {repeated:,}')
    print(f'Historical labeled pages: {len(historical):,}; current June pages: {len(current):,}')
    return historical, current

def load_frames() -> tuple[pd.DataFrame, pd.DataFrame]:
    output = repo_root() / 'work' / 'outputs'
    history_path = output / 'lane2_history_2026-05-31.parquet'
    current_path = output / 'lane2_current_2026-06-30.parquet'
    if history_path.exists() and current_path.exists():
        con = duckdb.connect()
        history = con.sql(f"SELECT * FROM read_parquet('{_quote_path(history_path)}')").df()
        current = con.sql(f"SELECT * FROM read_parquet('{_quote_path(current_path)}')").df()
        print(f'Loaded cached frames: {len(history):,} historical, {len(current):,} current pages')
        return history, current
    history, current = build_frames()
    output.mkdir(parents=True, exist_ok=True)
    con = duckdb.connect()
    con.register('cache_history', history)
    con.register('cache_current', current)
    con.execute(f"COPY cache_history TO '{_quote_path(history_path)}' (FORMAT PARQUET)")
    con.execute(f"COPY cache_current TO '{_quote_path(current_path)}' (FORMAT PARQUET)")
    print('Saved local Parquet caches under work/outputs; dataset files are gitignored')
    return history, current


In [4]:
def grouped_split(frame: pd.DataFrame) -> tuple[np.ndarray, np.ndarray]:
    groups = frame['client_hash_id']
    if groups.nunique() < 2:
        raise ValueError('A grouped split needs at least two clients')
    train, test = next(GroupShuffleSplit(n_splits=1, test_size=0.25,
                                         random_state=42).split(frame, groups=groups))
    assert set(groups.iloc[train]).isdisjoint(set(groups.iloc[test]))
    return train, test

def baseline_settings(frame: pd.DataFrame, train: np.ndarray) -> tuple[float, float]:
    part = frame.iloc[train]
    return float(part['prior30_impressions'].median()), float(part['prior30_ctr'].quantile(0.25))

def baseline_rank(frame: pd.DataFrame, settings: tuple[float, float]) -> pd.DataFrame:
    visibility_min, low_ctr_max = settings
    visible = frame['prior30_impressions'] >= visibility_min
    low_ctr = frame['prior30_ctr'] <= low_ctr_max
    striking = frame['prior30_avg_position'].between(11, 20, inclusive='both')
    result = frame[['client_hash_id', 'content_hash_id', 'prior30_impressions',
                    'prior30_clicks', 'prior30_ctr', 'prior30_avg_position', 'position_missing']].copy()
    result['baseline_score'] = (visible.astype(int) * (2 + low_ctr.astype(int) + striking.astype(int))
                                + 0.01 * np.log1p(frame['prior30_impressions']))
    result['reason_code'] = np.select(
        [visible & low_ctr & striking, visible & low_ctr, visible & striking, visible],
        ['visible_low_ctr_striking', 'visible_low_ctr', 'visible_striking', 'visible_page'],
        default='low_visibility_review_later',
    )
    return result

def precision_at_k(scores, labels, k: int) -> tuple[float, int]:
    values = np.asarray(scores, dtype=float)
    truth = np.asarray(labels, dtype=int)
    if len(values) != len(truth) or len(values) == 0:
        raise ValueError('Scores and labels need the same nonzero length')
    used = min(k, len(values))
    order = np.argsort(-values, kind='stable')[:used]
    return float(truth[order].mean()), used


In [5]:
from pathlib import Path
import numpy as np
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

history, current = load_frames()
feature_columns = list(FEATURE_COLUMNS)
X = history[feature_columns]
y = history.future_decline_label.astype(int)
assert y.nunique() == 2, 'The eligible history needs both decline and non-decline examples'
model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000, random_state=42))
print('Method: scaled logistic regression; target: observed June decline')
print('Model inputs:', ', '.join(feature_columns))

Loaded cached frames: 102,532 historical, 96,678 current pages
Method: scaled logistic regression; target: observed June decline
Model inputs: log_prior30_impressions, log_prior30_clicks, prior30_ctr, prior30_avg_position_filled, position_missing


## 2. Split design

A random page split can put pages from the same client on both sides. The main check holds out whole clients with the same seed and test fraction used in the Week 4 baseline. Both the score and its comparison are computed only on held-out clients. One May-to-June cutoff does not test whether the method stays accurate in a later season.

In [6]:
train_idx, test_idx = grouped_split(history)
train_clients = set(history.client_hash_id.iloc[train_idx])
test_clients = set(history.client_hash_id.iloc[test_idx])
assert train_clients.isdisjoint(test_clients)
settings = baseline_settings(history, train_idx)
print(f'Train pages: {len(train_idx):,}; test pages: {len(test_idx):,}')
print(f'Train clients: {len(train_clients):,}; held-out clients: {len(test_clients):,}')
print(f'Train decline rate: {y.iloc[train_idx].mean():.1%}; '
      f'test decline rate: {y.iloc[test_idx].mean():.1%}')

Train pages: 91,769; test pages: 10,763
Train clients: 31; held-out clients: 11
Train decline rate: 65.8%; test decline rate: 65.5%


## 3. Model compared with the baseline

I rank the same held-out clients with the rule and the model, then compare precision at 20 and 50 with the June decline base rate. I also compare model accuracy with always choosing the majority class. If the model does not beat these checks, I keep the simpler rule for the review queue. The saved model scores are an experiment, not the chosen action queue. July outcomes are not available to test a later cutoff.


In [7]:
model.fit(X.iloc[train_idx], y.iloc[train_idx])
model_scores = model.predict_proba(X.iloc[test_idx])[:, 1]
baseline_scores = baseline_rank(history, settings).baseline_score.iloc[test_idx]
test_labels = y.iloc[test_idx]
print(f'Held-out decline base rate: {test_labels.mean():.1%}')
for k in (20, 50):
    base_p, used = precision_at_k(baseline_scores, test_labels, k)
    model_p, _ = precision_at_k(model_scores, test_labels, k)
    print(f'Precision at {used}: baseline {base_p:.1%}; model {model_p:.1%}')
majority_class = int(y.iloc[train_idx].mean() >= 0.5)
majority_accuracy = accuracy_score(test_labels, np.full(len(test_idx), majority_class))
model_accuracy = accuracy_score(test_labels, model_scores >= 0.5)
baseline_p20, _ = precision_at_k(baseline_scores, test_labels, 20)
model_p20, _ = precision_at_k(model_scores, test_labels, 20)
print(f'Majority accuracy: {majority_accuracy:.1%}')
print(f'Model accuracy: {model_accuracy:.1%}')
if model_p20 <= baseline_p20 or model_accuracy <= majority_accuracy:
    print('Decision: keep the simple rule for the review queue; the model did not beat both checks')
else:
    print('Decision: the model looks promising on this split; test another time window before use')

final_model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000, random_state=42))
final_model.fit(X, y)
model_queue = baseline_rank(current, settings)
model_queue['model_score'] = final_model.predict_proba(current[feature_columns])[:, 1]
model_queue = model_queue.sort_values('model_score', ascending=False, kind='stable')
output_dir = repo / 'work' / 'outputs'
output_dir.mkdir(parents=True, exist_ok=True)
model_path = output_dir / 'model_action_score.csv'
model_queue.to_csv(model_path, index=False)
print(f'Experimental June model scores: {len(model_queue):,} pages; saved at {model_path}')

Held-out decline base rate: 65.5%
Precision at 20: baseline 95.0%; model 80.0%
Precision at 50: baseline 92.0%; model 82.0%
Majority accuracy: 65.5%
Model accuracy: 63.7%
Decision: keep the simple rule for the review queue; the model did not beat both checks
Experimental June model scores: 96,678 pages; saved at D:\flyrank-ml-internship-starter\work\outputs\model_action_score.csv


## 4. Errors and interpretation

The error check counts missed declines and false alarms on held-out clients. It also checks whether errors cluster among pages with lower May impressions or missing position. Coefficients show what the fitted model uses, but they are associations within this feature set. They do not say that changing one metric would change future traffic.

In [8]:
weights = model.named_steps['logisticregression'].coef_[0]
for name, weight in sorted(zip(feature_columns, weights),
                           key=lambda pair: abs(pair[1]), reverse=True):
    print(f'{name}: standardized coefficient {weight:+.3f}')
predicted = (model_scores >= 0.5).astype(int)
truth = test_labels.to_numpy()
print(f'Missed declines: {int(((predicted == 0) & (truth == 1)).sum()):,}')
print(f'False alarms: {int(((predicted == 1) & (truth == 0)).sum()):,}')
error_frame = history.iloc[test_idx].copy()
error_frame['wrong'] = predicted != truth
median_impressions = history.prior30_impressions.iloc[train_idx].median()
for label, mask in (
    ('lower May impressions', error_frame.prior30_impressions < median_impressions),
    ('higher May impressions', error_frame.prior30_impressions >= median_impressions),
    ('missing average position', error_frame.position_missing == 1),
):
    n = int(mask.sum())
    if n >= 30:
        print(f'{label}: {n:,} held-out pages; error rate {error_frame.loc[mask, "wrong"].mean():.1%}')
    else:
        print(f'{label}: {n:,} pages; too few for a stable error rate')
print('No client or page identifiers are displayed in this error check')

log_prior30_clicks: standardized coefficient -0.567
log_prior30_impressions: standardized coefficient +0.452
prior30_ctr: standardized coefficient -0.042
prior30_avg_position_filled: standardized coefficient +0.021
position_missing: standardized coefficient +0.000
Missed declines: 553
False alarms: 3,357
lower May impressions: 5,636 held-out pages; error rate 39.4%
higher May impressions: 5,127 held-out pages; error rate 33.0%
missing average position: 0 pages; too few for a stable error rate
No client or page identifiers are displayed in this error check


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under work/notebooks/ — then submit your repo URL on the card. Done.